# 6 · Knowledge graphs

Knowledge-aware models score from a graph of facts about the items as well as
from the interactions. This guide reads the LastFM data and the Freebase graph
released with KGCN, and covers:

- the two files `reader.knowledge` reads (triples and an item → entity
  alignment), through the Python API and through a configuration;
- **coverage**: only part of the catalogue is aligned to an entity, what WarpRec
  reports and warns about, what each model does with an uncovered item, and how
  to restrict the catalogue to the covered items;
- the `KnowledgeGraph` a `Dataset` carries: re-indexing, the item → entity
  alignment, neighbourhoods, the adjacency matrix and the first- and
  second-order item features the feature-based models read;
- a short training run of KaHFM next to BPR through the design pipeline.

**Requirements:** `pip install warprec jupyter`. Runs in one to two minutes on
a laptop CPU, most of it training; it downloads the KGCN music files (2 MB) into
`guides/data/`.

In [1]:
import sys
import warnings
from pathlib import Path

import pandas as pd
import torch
from loguru import logger

sys.path.insert(0, "..")
from guide_data import kgcn_music

RUNS = Path("../runs/06-knowledge-graphs")
RUNS.mkdir(parents=True, exist_ok=True)

# PyTorch warns once about the sparse tensors the graph is turned into; it says
# nothing about the data.
warnings.filterwarnings("ignore", message="Sparse invariant checks")

## The files

`kgcn_music()` downloads three files from the KGCN repository. The listening
data is HetRec's LastFM: one row per user and artist, with a play count.

In [2]:
folder = kgcn_music()
plays = pd.read_csv(folder / "user_artists.dat", sep="\t")
print(
    f"{len(plays):,} interactions, {plays.userID.nunique():,} users, "
    f"{plays.artistID.nunique():,} artists"
)
plays.head(3)

92,834 interactions, 1,892 users, 17,632 artists


,userID,artistID,weight
0,2,51,13883
1,2,52,11690
2,2,53,11351


The graph comes as the two files `reader.knowledge` expects, already in its
default format (tab separated, no header). `kg.txt` holds the facts as
`(head, relation, tail)`; `item_index2entity_id.txt` says which entity each
artist **is**, as `(artistID, entity)`. The entities are anonymous integers
(Freebase identifiers were replaced by KGCN); the relations keep their Freebase
names.

In [3]:
triples = pd.read_csv(folder / "kg.txt", sep="\t", names=["head", "relation", "tail"])
links = pd.read_csv(
    folder / "item_index2entity_id.txt", sep="\t", names=["item_id", "entity_id"]
)
print(f"{len(triples):,} triples, {triples.relation.nunique()} relations")
print(f"{len(links):,} aligned artists")
pd.concat([triples.head(3), links.head(3)], axis=1)

15,518 triples, 60 relations
3,846 aligned artists


,head,relation,tail,item_id,entity_id
0,2086,music.artist.origin,3846,2,0
1,1601,film.person_or_entity_appearing_in_film.film,3847,3,1
2,3355,film.actor.film,3848,4,2


Nothing needs converting, but two properties are worth checking before
trusting an alignment: each item is aligned to one entity, and every aligned
entity appears in at least one fact (WarpRec warns about the ones that do not,
because those items would carry no facts).

In [4]:
entities = set(triples["head"]) | set(triples["tail"])
print("one entity per artist:", links.item_id.is_unique)
print(
    "aligned entities missing from the triples:",
    (~links.entity_id.isin(entities)).sum(),
)

one entity per artist: True
aligned entities missing from the triples: 0


## Coverage

The alignment covers a fraction of the artists people listened to, and the
covered artists are barely more popular than the rest: their share of the
interactions is close to their share of the catalogue.

In [5]:
covered = plays.artistID.isin(links.item_id)
pd.DataFrame(
    {
        "all": [plays.artistID.nunique(), len(plays)],
        "aligned to an entity": [plays.artistID[covered].nunique(), covered.sum()],
    },
    index=["artists", "interactions"],
).assign(share=lambda t: (t.iloc[:, 1] / t.iloc[:, 0]).round(3))

,all,aligned to an entity,share
artists,17632,3846,0.218
interactions,92834,21173,0.228


## Reading the graph through the API

A `Dataset` takes the two frames as `knowledge_data` and `knowledge_links`.
Their column names default to `head`, `relation`, `tail` and `item_id`,
`entity_id`; `knowledge_labels` maps other names onto those. The reader reads
them like any headerless file.

In [6]:
from warprec.data import Dataset
from warprec.data.reader import LocalReader
from warprec.data.schema import SplitSpec
from warprec.data.splitting import Splitter

reader = LocalReader()
kg_triples = reader.read_tabular(
    local_path=str(folder / "kg.txt"),
    sep="\t",
    header=False,
    column_names=["head", "relation", "tail"],
)
kg_links = reader.read_tabular(
    local_path=str(folder / "item_index2entity_id.txt"),
    sep="\t",
    header=False,
    column_names=["item_id", "entity_id"],
)
interactions = reader.read_tabular(
    local_path=str(folder / "user_artists.dat"), sep="\t"
).rename({"userID": "user_id", "artistID": "item_id"})

train, _, test = Splitter().split_transaction(
    interactions, test=SplitSpec(strategy="random_holdout", ratio=0.2, seed=42)
)

08-10-2026 12:24:05 - WarpRec - 📝 Starting test splitting process with random_holdout splitting strategy.


08-10-2026 12:24:05 - WarpRec - 📝 Test splitting completed in : 0.01s


08-10-2026 12:24:05 - WarpRec - ✅ Splitting process over in 0.01s.


08-10-2026 12:24:05 - WarpRec - 📢 Test set was not aligned with the training set. Filtered out 2313 transactions.


The graph is aligned once the item mapping is settled, that is against the
items of the **training** split. The log reports the size of the graph and how
many of those items have an entity.

In [7]:
dataset = Dataset(
    train_data=train,
    eval_data=test,
    knowledge_data=kg_triples,
    knowledge_links=kg_links,
)

08-10-2026 12:24:05 - WarpRec - 📝 --------------------------------------Knowledge graph--------------------------------------


08-10-2026 12:24:05 - WarpRec - 📊 Triples: 15518      Entities: 9366      Relations: 60      Items with an entity: 3472/15431


08-10-2026 12:24:05 - WarpRec - 📝 -------------------------------------------------------------------------------------------


08-10-2026 12:24:05 - WarpRec - 📝 --------------------------------Train set---------------------------------


08-10-2026 12:24:05 - WarpRec - 📊 Number of users: 1889      Number of items: 15431      Transactions: 74267


08-10-2026 12:24:05 - WarpRec - 📝 --------------------------------------------------------------------------


08-10-2026 12:24:05 - WarpRec - 📝 ---------------------------------Test set--------------------------------


08-10-2026 12:24:05 - WarpRec - 📊 Number of users: 1873      Number of items: 4168      Transactions: 16254


08-10-2026 12:24:05 - WarpRec - 📝 -------------------------------------------------------------------------


Aligned artists that never occur in training are not counted: the alignment is
written for the whole graph, so rows naming items outside the catalogue are
skipped silently.

## The same thing from a configuration

In a configuration the two files are the `reader.knowledge` section. Both
paths are required, and the section's defaults (`sep: "\t"`, `header: false`,
`column_names: [head, relation, tail]`,
`link_column_names: [item_id, entity_id]`) match this release. The item column
of the alignment is read with the type of the interactions' item column, so
the two match.

In [8]:
print(Path("configs/knowledge.yml").read_text())

# The LastFM listening data with the knowledge graph KGCN released for it.
# Paths are relative to the guide's folder.
reader:
  loading_strategy: dataset
  data_type: transaction
  reading_method: local
  local_path: ../data/kgcn-music/user_artists.dat
  rating_type: implicit
  labels:
    user_id_label: userID
    item_id_label: artistID
  knowledge:
    # (head, relation, tail), tab separated, no header: the defaults.
    local_path: ../data/kgcn-music/kg.txt
    # (item, entity), same format.
    link_path: ../data/kgcn-music/item_index2entity_id.txt
splitter:
  test_splitting:
    strategy: random_holdout
    ratio: 0.2
    seed: 42
models:
  Pop: {}
evaluation:
  top_k: [10]
  metrics: [nDCG]



In [9]:
from warprec.common.initialize import initialize_datasets
from warprec.data.reader import ReaderFactory
from warprec.utils.callback import WarpRecCallback
from warprec.utils.config.config import load_design_configuration

logger.disable("warprec")
config = load_design_configuration("configs/knowledge.yml")
full, _, _ = initialize_datasets(
    ReaderFactory.get_reader(config), WarpRecCallback(), config
)
kg = full.knowledge
items_with_entity = int((kg.get_item_entities() >= 0).sum())
print(
    f"{items_with_entity:,} of {full.info()['n_items']:,} training items "
    "have an entity; same as the API:",
    torch.equal(kg.get_item_entities(), dataset.knowledge.get_item_entities()),
)

3,472 of 15,431 training items have an entity; same as the API: True


## The `KnowledgeGraph`

`dataset.knowledge` holds the graph in index space. `info()` gains two keys,
which the knowledge models size their embedding tables from:

In [10]:
{k: v for k, v in full.info().items() if not isinstance(v, dict)}

{'n_items': 15431,
 'n_users': 1889,
 'n_features': 0,
 'n_entities': 9366,
 'n_relations': 60}

Entities and relations are re-indexed from 0 in the sorted order of their raw
identifiers, heads and tails sharing one vocabulary. This release already
numbers its entities 0…9365, so an entity's index is its identifier; the
relations, which are strings, become 0…59 in alphabetical order.
`get_triples()` returns the facts as three long tensors.

In [11]:
heads, relations, tails = kg.get_triples()
relation_names = sorted(triples.relation.unique())
print("triples:", len(kg), "| entities, relations:", kg.get_dims())
print("entity index == raw id:", torch.equal(heads, torch.as_tensor(triples["head"])))
print(
    "first fact:",
    (heads[0].item(), relations[0].item(), tails[0].item()),
    "->",
    (triples["head"][0], relation_names[relations[0]], triples["tail"][0]),
)

triples: 15518 | entities, relations: (9366, 60)


entity index == raw id: True
first fact: (2086, 44, 3846) -> (np.int64(2086), 'music.artist.origin', np.int64(3846))


`get_item_entities()` gives, for every item index, the entity it stands for,
or `-1`. Indexed with the dataset's item mapping:

In [12]:
_, item_map = full.get_mappings()
item_entity = kg.get_item_entities()
pd.DataFrame(
    {
        "artistID": [2, 3, 6, 9],
        "item index": [item_map[a] for a in (2, 3, 6, 9)],
        "entity": [item_entity[item_map[a]].item() for a in (2, 3, 6, 9)],
    }
)

,artistID,item index,entity
0,2,1,0
1,3,2,1
2,6,5,3
3,9,8,4


### Neighbourhoods

`neighbour_index()` returns the graph as adjacency lists in CSR form:
`offsets`, and the `neighbours` and `relations` arrays, so that entity `e`'s
neighbours are `neighbours[offsets[e]:offsets[e + 1]]`. A fact is read from
both ends, so the lists hold every triple twice. This is what the propagating
models (KGCN, KGIN, RippleNet) walk.

In [13]:
offsets, neighbours, via = kg.neighbour_index()
degree = offsets[1:] - offsets[:-1]
item_degree = degree[item_entity[item_entity >= 0]].float()
print("neighbour entries:", len(neighbours), "= 2 x", len(kg))
print(
    f"neighbours of an item entity: median {item_degree.median():.0f}, "
    f"mean {item_degree.mean():.1f}, max {item_degree.max():.0f}"
)

neighbour entries: 31036 = 2 x 15518
neighbours of an item entity: median 1, mean 4.1, max 943


Many items sit on one or two facts. `sample_neighbours(k, generator)` turns
the ragged lists into a rectangle of `k` neighbours per entity, sampling with
replacement where an entity has fewer than `k` (KGCN does this once, before
training). Entity 0 has a single neighbour, so it is drawn four times; the
second entity below has more than four, so four of them are drawn. The
`generator` makes the draw reproducible.

In [14]:
sampled, sampled_relations = kg.sample_neighbours(4, torch.Generator().manual_seed(42))
print(sampled.shape)
for e in (0, int((degree > 4).nonzero()[0])):
    lists = neighbours[offsets[e] : offsets[e + 1]].tolist()
    print(f"entity {e}: neighbours {lists[:6]} -> sampled {sampled[e].tolist()}")

torch.Size([9366, 4])
entity 0: neighbours [4454] -> sampled [4454, 4454, 4454, 4454]
entity 30: neighbours [4227, 4227, 4498, 4764, 6696, 4221] -> sampled [6696, 4498, 4764, 4227]


`adjacency()` is the same graph as a sparse `entities × entities` matrix, head
to tail, for the models that propagate by matrix product (KGAT). It ignores the
relation: facts joining the same two entities through different relations are
summed into one entry. Here that is common, because the graph states an
artist's film roles twice (`film.actor.film` and
`film.person_or_entity_appearing_in_film.film`).

In [15]:
adjacency = kg.adjacency()
print(
    adjacency.shape,
    "| stored:",
    adjacency._nnz(),
    "of",
    len(kg),
    "facts | largest entry:",
    adjacency.values().max().item(),
)

torch.Size([9366, 9366]) | stored: 9807 of 15518 facts | largest entry: 6.0


`values=` weights each triple and `size=`/`offset=` leave room for extra rows,
which is how KGAT prepends its users to the entity graph.

### Item features

The feature-based models (KaHFM, KGFlex) read the graph as an attribute table.
`item_features(order, min_items)` returns a sparse binary `items × features`
matrix and the label of each column, in the graph's own identifiers. A
first-order feature is a `(relation, tail)` pair leaving the item's entity; a
second-order one follows one more fact, `(relation, relation, tail)`. A feature
is kept only if at least `min_items` items carry it; KaHFM and KGFlex expose
this as `min_feature_items`.

In [16]:
rows = []
for order in (1, 2):
    for min_items in (1, 2, 5, 10):
        matrix, labels = kg.item_features(order=order, min_items=min_items)
        rows.append(
            {
                "order": order,
                "min_items": min_items,
                "features": matrix.shape[1],
                "items with a feature": int((matrix.to_dense().sum(dim=1) > 0).sum()),
            }
        )
pd.DataFrame(rows)

,order,min_items,features,items with a feature
0,1,1,10340,3472
1,1,2,1451,3012
2,1,5,215,2048
3,1,10,79,1509
4,2,1,2432,26
5,2,2,942,4
6,2,5,0,0
7,2,10,0,0


8,889 of the 10,340 first-order features are carried by a single artist, so
requiring two items already removes most columns. Second-order features are
almost absent (26 items carry one at all): every head of this graph is an
artist, so a second step needs a tail that is itself an artist, which is rare. KGFlex's
`second_order_limit` would find nothing to keep on this graph. The labels:

In [17]:
matrix, labels = kg.item_features(order=1, min_items=10)
counts = matrix.to_dense().sum(dim=0)
pd.DataFrame(labels, columns=["relation", "tail entity"]).assign(
    items=counts.int()
).sort_values("items", ascending=False).head(5)

,relation,tail entity,items
7,music.artist.origin,3864,119
23,music.artist.origin,4057,103
64,music.musician.instruments_played,4349,59
44,music.artist.origin,4454,54
40,music.artist.origin,4362,53


## What a model does with an uncovered item

Every knowledge-aware model maps an uncovered item to a **padding entity**, one
row past the real ones, whose embedding is held at zero. What that means
depends on the model:

- **KGAT, KGIN, KGCN, RippleNet** represent an item *by* its entity. All
  uncovered items share the padding row, so the model cannot tell them apart.
- **CKE** adds the entity embedding to a separate collaborative item
  embedding; an uncovered item keeps the latter and behaves as in plain MF.
- **KaHFM** starts an uncovered item's factors at zero, so before training only
  its bias scores it; **KGFlex** scores it zero.

When fewer than half the items are covered, building any of these models logs
a warning. Here CKE is built on the full catalogue (only the constructor, no
training); its `item_entity` buffer shows uncovered items pointing at entity
9366, the padding row.

In [18]:
from warprec.utils.registry import model_registry

logger.enable("warprec")
cke = model_registry.get(
    "CKE",
    params={
        "embedding_size": 64,
        "kg_embedding_size": 64,
        "reg_weight": 0.001,
        "kg_reg_weight": 0.001,
        "batch_size": 1024,
        "epochs": 1,
        "learning_rate": 0.005,
    },
    info=full.info(),
    knowledge=full.knowledge,
    seed=42,
)
logger.disable("warprec")
print("entity of items 0-7:", cke.item_entity[:8].tolist())

08-10-2026 12:24:05 - WarpRec - 📢 The knowledge graph covers 3472 of 15431 items. Models that read an item as its entity score the uncovered ones as one and the same, so a graph this partial will hurt them. Consider restricting the catalogue to the items it covers.


entity of items 0-7: [9366, 0, 1, 2, 9366, 3, 9366, 9366]


The warning is about the representation, not about correctness: nothing fails,
but a propagation model on this catalogue would represent more than three
quarters of it by one shared vector. The usual practice, and what KGCN's own
preprocessing of this data does, is to keep only the covered items.

## Restricting the catalogue to covered items

The `DropItem` filter (guide 2) removes a list of item ids, which would mean
listing the 13,786 uncovered artists in the configuration. Filtering the file
once is simpler, and the result is an ordinary interaction file:

In [19]:
plays[covered].to_csv(RUNS / "user_artists_covered.tsv", sep="\t", index=False)
print(f"{covered.sum():,} interactions kept, {plays[covered].userID.nunique():,} users")

21,173 interactions kept, 1,872 users


The configuration for the training run reads that file with the same
`reader.knowledge` section, and trains BPR and KaHFM on it.
`optimization.num_workers: 0` loads batches in the notebook's own process
instead of starting data-loader worker processes.

In [20]:
print(Path("configs/train.yml").read_text())

# Only the artists the graph covers, a knowledge model next to BPR.
reader:
  loading_strategy: dataset
  data_type: transaction
  reading_method: local
  local_path: ../runs/06-knowledge-graphs/user_artists_covered.tsv
  rating_type: implicit
  labels:
    user_id_label: userID
    item_id_label: artistID
  knowledge:
    local_path: ../data/kgcn-music/kg.txt
    link_path: ../data/kgcn-music/item_index2entity_id.txt
splitter:
  test_splitting:
    strategy: random_holdout
    ratio: 0.2
    seed: 42
models:
  BPR:
    embedding_size: 64
    reg_weight: 0.001
    batch_size: 1024
    epochs: 20
    learning_rate: 0.005
    optimization:
      num_workers: 0
  KaHFM:
    min_feature_items: 2
    reg_weight: 0.001
    bias_reg_weight: 0.0
    batch_size: 1024
    epochs: 20
    learning_rate: 0.005
    optimization:
      num_workers: 0
evaluation:
  top_k: [10, 20]
  metrics: [nDCG, Recall, Precision]



Every training item now has an entity:

In [21]:
config = load_design_configuration("configs/train.yml")
covered_ds, _, _ = initialize_datasets(
    ReaderFactory.get_reader(config), WarpRecCallback(), config
)
entities_of_items = covered_ds.knowledge.get_item_entities()
print(
    f"{int((entities_of_items >= 0).sum()):,} of {len(entities_of_items):,} "
    "items have an entity"
)

3,436 of 3,436 items have an entity


KaHFM's width is the number of features it keeps, one factor per feature, and
each factor keeps its label in `feature_labels`. Built on the covered dataset
(again only the constructor), it has one factor per first-order feature carried
by at least two artists:

In [22]:
kahfm = model_registry.get(
    "KaHFM",
    params=config.models["KaHFM"],
    info=covered_ds.info(),
    interactions=covered_ds.train_set,
    knowledge=covered_ds.knowledge,
    seed=42,
)
starting = kahfm.item_factors.weight[:-1]
print("factors:", starting.shape[1])
print("items starting at zero:", int((starting.abs().sum(dim=1) == 0).sum()))
print("first labels:", kahfm.feature_labels[:3])

factors: 1418
items starting at zero: 451
first labels: [('event.agent.performance', 4035), ('event.agent.performance', 4099), ('event.agent.performance', 4251)]


## Training KaHFM next to BPR

`design_pipeline` runs a configuration end to end: it reads, splits, builds
each model in `models` with the dataset's knowledge graph, trains it and logs
the test metrics (guide 9 covers the pipelines). The log is on so the results
tables show; Lightning adds a summary of each model's parameters. Everything
runs in the notebook's own process.

In [23]:
from warprec.pipelines.design import design_pipeline

logger.enable("warprec")
design_pipeline("configs/train.yml")

08-10-2026 12:24:06 - WarpRec - 📝 Starting the Design Pipeline.


08-10-2026 12:24:06 - WarpRec - 📝 Reading design configuration file in: configs/train.yml


08-10-2026 12:24:06 - WarpRec - 📝 Reading process completed correctly.


08-10-2026 12:24:06 - WarpRec - 📝 Starting test splitting process with random_holdout splitting strategy.


08-10-2026 12:24:06 - WarpRec - 📝 Test splitting completed in : 0.00s


...

08-10-2026 12:24:26 - WarpRec - 📝 | Top@10   | 0.163848 | 0.24138  |   0.0546749 |


08-10-2026 12:24:26 - WarpRec - 📝 +----------+----------+----------+-------------+


08-10-2026 12:24:26 - WarpRec - 📝 | Top@20   | 0.194478 | 0.337674 |   0.0385759 |


08-10-2026 12:24:26 - WarpRec - 📝 +----------+----------+----------+-------------+


08-10-2026 12:24:26 - WarpRec - ✅ Design pipeline executed successfully. WarpRec is shutting down.


Read the two `Test` tables. With these settings KaHFM lands a little below
BPR on every metric. That is in line with this graph: with
`min_feature_items: 2`, 451 of the 3,436 artists (about one in eight) start
with all-zero factors, learned from the interactions alone with no feature to
start from. What KaHFM offers in exchange is that each of its factors is a named
fact. Tuning both models (guide 10) is what a real comparison needs.

## Summary of the `reader.knowledge` section

| key | what it does |
|---|---|
| `local_path` | the `(head, relation, tail)` triples |
| `link_path` | the `(item, entity)` alignment; required |
| `sep`, `header` | format of both files; default `"\t"` and `false` |
| `column_names` | names of the three triple columns, in order; default `[head, relation, tail]` |
| `link_column_names` | names of the two alignment columns; default `[item_id, entity_id]` |
| `file_format` | must be `tabular` |

The models' own keys used here are `min_feature_items` (KaHFM, KGFlex) and
the embedding widths of CKE. See the
[Reader configuration](https://warprec.readthedocs.io/en/latest/configuration/reader/)
page for the section, and the
[Knowledge-aware recommenders](https://warprec.readthedocs.io/en/latest/recommenders/knowledge/)
page for every model and how it treats uncovered items.